# 59 — English transfer evaluation (existing models)

**Purpose:** Measure **cross-lingual / domain transfer** for **already trained** checkpoints on canonical English artifacts from **32**, optionally including the softer slice from **54**. **No training** here.

**Shared pipeline:** `notebooks/english/english_eval_common.run_english_eval_pipeline` with custom output roots.

**Outputs:** `notebooks/results/english_transfer_eval/<model_run_id>/` and `figures/english_transfer_eval/<model_run_id>/`.

Run once per model (or set `ENGLISH_TRANSFER_MODEL_RUN_ID` before executing the inference cell).


In [1]:
import os
import sys
from pathlib import Path

_CAND = [
    Path.cwd().resolve(),
    Path.cwd().resolve() / "english",
    Path.cwd().resolve() / "notebooks" / "english",
]
ENGLISH_DIR = next((p for p in _CAND if (p / "english_eval_common.py").exists()), None)
if ENGLISH_DIR is None:
    raise FileNotFoundError("english_eval_common.py not found")
if str(ENGLISH_DIR) not in sys.path:
    sys.path.insert(0, str(ENGLISH_DIR))

MODEL_RUN_ID = os.environ.get("ENGLISH_TRANSFER_MODEL_RUN_ID", "bert_9classes_final")
print("MODEL_RUN_ID:", MODEL_RUN_ID)


MODEL_RUN_ID: bert_9classes_final


In [2]:
import json
from english_eval_common import run_english_eval_pipeline

summary = run_english_eval_pipeline(
    MODEL_RUN_ID,
    batch_size=16,
    max_length=128,
    skip_missing_csv=True,
    results_subdir="english_transfer_eval",
    figures_subdir="english_transfer_eval",
)

# Compact print: accuracy per dataset where available
brief = {}
for ds, block in summary.get("per_dataset", {}).items():
    if isinstance(block, dict) and "accuracy" in block:
        brief[ds] = {"accuracy": block["accuracy"], "macro_f1": block.get("macro_f1"), "n_evaluated": block.get("n_evaluated")}
print(json.dumps(brief, indent=2))


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2214.34it/s, Materializing param=classifier.weight]                                      


{
  "english_train": {
    "accuracy": 0.11214557765552265,
    "macro_f1": 0.05641677845838192,
    "n_evaluated": 2363
  },
  "english_val": {
    "accuracy": 0.10059171597633136,
    "macro_f1": 0.047514111516652364,
    "n_evaluated": 507
  },
  "english_test": {
    "accuracy": 0.11637080867850098,
    "macro_f1": 0.055912625805098926,
    "n_evaluated": 507
  },
  "english_soft_eval_slice": {
    "accuracy": 0.12261806130903065,
    "macro_f1": 0.05844804284681466,
    "n_evaluated": 2414
  },
  "english_base_eval_slice": {
    "accuracy": 0.142379679144385,
    "macro_f1": 0.0640497521400942,
    "n_evaluated": 1496
  }
}


## Key takeaways

1. **Transfer ranking:** Compare **accuracy / macro-F1** on `english_test` and the **soft slice** (`english_soft_eval_slice` when 54 has been run) across models using **separate result folders** per `MODEL_RUN_ID`.
2. **Gap vs main benchmark:** If **in-domain** baselines are recorded elsewhere, the metrics here quantify **degradation** on English; missing CSVs are **skipped** intentionally when optional artifacts are absent.
3. **Story fit:** Stronger English numbers support a **generalization** narrative alongside **adversarial pair** stability (notebooks **56–58**) and **design-time** fairness probes — each answers a different failure mode.
